# Notebook 2 – Portfolio Analytics
This notebook loads the processed portfolio data and computes key financial risk and performance metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

portfolio = pd.read_csv("data/processed/portfolio_daily.csv", parse_dates=True, index_col=0)
returns = portfolio["Portfolio_Return"].dropna()

RISK_FREE_RATE = 0.06
TRADING_DAYS = 252


In [ ]:
annual_return = returns.mean()*TRADING_DAYS
annual_volatility = returns.std()*np.sqrt(TRADING_DAYS)
sharpe = (annual_return-RISK_FREE_RATE)/annual_volatility

downside = returns[returns<0]
sortino = (annual_return-RISK_FREE_RATE)/(downside.std()*np.sqrt(TRADING_DAYS))

growth = (1+returns).cumprod()
years = (growth.index[-1]-growth.index[0]).days/365.25
cagr = growth.iloc[-1]**(1/years)-1

print(f"CAGR: {cagr:.2%}")
print(f"Annual Return: {annual_return:.2%}")
print(f"Annual Volatility: {annual_volatility:.2%}")
print(f"Sharpe Ratio: {sharpe:.3f}")
print(f"Sortino Ratio: {sortino:.3f}")


In [ ]:
peak = growth.cummax()
drawdown = growth/peak - 1
max_dd = drawdown.min()

var95 = np.percentile(returns,5)
cvar95 = returns[returns<=var95].mean()

print(f"Maximum Drawdown: {max_dd:.2%}")
print(f"Historical VaR (95%): {var95:.2%}")
print(f"Historical CVaR (95%): {cvar95:.2%}")


In [ ]:
rolling_vol = returns.rolling(21).std()*np.sqrt(TRADING_DAYS)
rolling_sharpe = (returns.rolling(63).mean()*TRADING_DAYS-RISK_FREE_RATE)/(returns.rolling(63).std()*np.sqrt(TRADING_DAYS))

plt.figure(figsize=(12,4))
plt.plot(growth)
plt.title("Portfolio Growth")
plt.grid(True)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(drawdown)
plt.title("Drawdown")
plt.grid(True)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(rolling_vol)
plt.title("Rolling Volatility (21D)")
plt.grid(True)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(rolling_sharpe)
plt.title("Rolling Sharpe Ratio")
plt.grid(True)
plt.show()


In [ ]:
summary = pd.DataFrame({
    "Metric":[
        "CAGR","Annual Return","Annual Volatility",
        "Sharpe Ratio","Sortino Ratio",
        "Maximum Drawdown","VaR95","CVaR95"
    ],
    "Value":[
        cagr,annual_return,annual_volatility,
        sharpe,sortino,max_dd,var95,cvar95
    ]
})

summary.to_excel("output/Portfolio_Analytics.xlsx",index=False)
summary


## Notebook 3 Preview

The next notebook will create advanced ML features:

- Rolling returns
- Momentum
- Volatility features
- Beta
- Correlation
- RSI
- MACD
- ATR
- Bollinger Bands
- VIX features
- Market features

These features will feed the walk-forward ML backtesting engine.
